In [12]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

In [13]:
g = 9.81     # gravity (m/s^2)
L = 1.0      # length (m)
theta_0 = 0.2  # initial angle (rad)

t_min, t_max = 0.0, 10.0

In [14]:
class PINN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 16),
            nn.Tanh(),
            nn.Linear(16,16),
            nn.Tanh(),
            nn.Linear(16,1)
        )
    def forward(self, t):
        return self.net(t)

In [15]:
def pinn_loss(model, t):
    t.requires_grad = True

    theta = model(t)

    theta_t = torch.autograd.grad(
        theta, t, grad_outputs = torch.ones_like(theta), create_graph = True
    )[0]

    theta_tt = torch.autograd.grad(
        theta_t, t, grad_outputs=torch.ones_like(theta_t), create_graph=True
    )[0]

    physics_residual = theta_tt + (g / L) * theta

    physics_loss = torch.mean(physics_residual**2)

    theta_0_pred = model(torch.tensor([[0.0]]))
    theta_t_0_pred = theta_t[0]

    ic_loss = (theta_0_pred - theta_0)**2 + theta_t_0_pred**2

    return physics_loss + ic_loss

In [16]:
model = PINN()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 20000

for epoch in range(epochs+1):
    optimizer.zero_grad()

    t_train = torch.linspace(t_min, t_max, 100).reshape(-1, 1)
    loss = pinn_loss(model, t_train)

    loss.backward()
    optimizer.step()

    if epoch % 500 == 0:
        print(f"Epoch {epoch}, Loss = {loss.item():.6f}")

Epoch 0, Loss = 1.494862
Epoch 500, Loss = 0.035079
Epoch 1000, Loss = 0.032160
Epoch 1500, Loss = 0.029617
Epoch 2000, Loss = 0.025974
Epoch 2500, Loss = 0.021436
Epoch 3000, Loss = 0.020176
Epoch 3500, Loss = 0.020426
Epoch 4000, Loss = 0.019428
Epoch 4500, Loss = 0.018978
Epoch 5000, Loss = 0.018183
Epoch 5500, Loss = 0.016759
Epoch 6000, Loss = 0.016501
Epoch 6500, Loss = 0.016316
Epoch 7000, Loss = 0.016153
Epoch 7500, Loss = 0.016012
Epoch 8000, Loss = 0.016069
Epoch 8500, Loss = 0.015781
Epoch 9000, Loss = 0.015689
Epoch 9500, Loss = 0.016182
Epoch 10000, Loss = 0.015537
Epoch 10500, Loss = 0.015640
Epoch 11000, Loss = 0.015409
Epoch 11500, Loss = 0.015352
Epoch 12000, Loss = 0.015296
Epoch 12500, Loss = 0.015241
Epoch 13000, Loss = 0.015191
Epoch 13500, Loss = 0.019942
Epoch 14000, Loss = 0.015092
Epoch 14500, Loss = 0.015046
Epoch 15000, Loss = 0.015001
Epoch 15500, Loss = 0.014956
Epoch 16000, Loss = 0.014915
Epoch 16500, Loss = 0.014874
Epoch 17000, Loss = 0.014835
Epoch 175